# MCBOMs Worked ExampleA single-segment safety-benefit derivation, from raw severity counts through tothe optimization decision. This is the simplest end-to-end demonstration of theframework: one segment, one alternative, full visibility into every step of thebenefit calculation.The same instance is also implemented in `models/ampl/01_worked_example.mod`and `models/gams/01_worked_example.gms` — the numbers should match to the cent.

## 1. Install MCBOMsThe framework is installed directly from the public TTI repository.

In [ ]:
!pip install --quiet 'git+https://github.com/ttitamu/mcboms-optimization.git#egg=mcboms[pulp]'

## 2. InputsA 2-mile rural two-lane segment with the following observed and projectedcharacteristics. The treatment under consideration is centerline rumble strips,which has a documented Crash Modification Factor (CMF) for total crashes.

In [ ]:
segment_length_miles = 2.0analysis_horizon_years = 20discount_rate = 0.07# Observed crash history, by KABCO severity, annualized.crashes_per_year = {    "K": 0.05,   # Fatal    "A": 0.20,   # Incapacitating injury    "B": 0.40,   # Non-incapacitating injury    "C": 0.60,   # Possible injury    "O": 1.00,   # Property damage only}# USDOT BCA Guidance May 2025 unit costs (2024 dollars), per crash.crash_cost_by_severity = {    "K": 13_700_000,    "A":    843_000,    "B":    278_000,    "C":    176_000,    "O":     14_700,}# Treatment.cmf_total_crashes = 0.85    # 15% reduction in total crashes (FHWA CMF Clearinghouse)treatment_cost   = 30_000   # $/mile, materials + striping

## 3. Annual safety benefitAnnual benefit = Σ (crashes_per_year × crash_cost) × (1 − CMF), summed acrossseverity levels and scaled by segment length.

In [ ]:
annual_crash_cost = sum(    crashes_per_year[k] * crash_cost_by_severity[k] for k in crashes_per_year)crash_reduction_factor = 1 - cmf_total_crashesannual_safety_benefit = annual_crash_cost * crash_reduction_factorprint(f"Annual crash cost (no treatment): ${annual_crash_cost:>15,.0f}")print(f"Crash reduction factor:          {crash_reduction_factor:>16.2%}")print(f"Annual safety benefit:           ${annual_safety_benefit:>15,.0f}")

## 4. Present-value benefit and costApply a uniform-series present-worth factor over the 20-year horizon at 7%.

In [ ]:
def uspw_factor(rate: float, years: int) -> float:    """Uniform-series present-worth factor."""    return ((1 + rate) ** years - 1) / (rate * (1 + rate) ** years)pwf = uspw_factor(discount_rate, analysis_horizon_years)pv_safety_benefit = annual_safety_benefit * pwfpv_treatment_cost = treatment_cost * segment_length_miles  # one-time installationprint(f"Present-worth factor (7%, 20yr): {pwf:.4f}")print(f"PV of safety benefit:            ${pv_safety_benefit:>15,.0f}")print(f"Treatment cost:                  ${pv_treatment_cost:>15,.0f}")print(f"Net benefit:                     ${pv_safety_benefit - pv_treatment_cost:>15,.0f}")

## 5. Run through the optimizerA single-segment problem is trivially solvable by hand, but we run it throughthe optimizer to demonstrate the API and verify that the framework produces thesame answer as the manual derivation.

In [ ]:
import pandas as pdfrom mcboms.core.optimizer import Optimizersites = pd.DataFrame({    "site_id": [1],    "facility_type": ["rural_2lane"],})alternatives = pd.DataFrame({    "site_id":      [1, 1],    "alt_id":       [0, 1],    "description":  ["do_nothing", "centerline_rumble_strips"],    "total_cost":   [0, pv_treatment_cost],    "total_benefit":[0, pv_safety_benefit],})# Budget large enough that the treatment is never the binding constraint.opt = Optimizer(sites=sites, alternatives=alternatives, budget=1_000_000)result = opt.solve()  # auto-detects gurobi or pulpprint(result.summary())

## 6. Cross-checkThe optimizer should select the rumble-strip alternative (alt_id=1) and reporta net benefit equal to `pv_safety_benefit - pv_treatment_cost`.

In [ ]:
manual_net = pv_safety_benefit - pv_treatment_costoptimizer_net = result.net_benefitprint(f"Manual derivation:   ${manual_net:>15,.0f}")print(f"Optimizer result:    ${optimizer_net:>15,.0f}")print(f"Match (within $1):   {abs(manual_net - optimizer_net) < 1.0}")